In [ ]:
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib
import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediData, merge_data_species
from JunoPy.index import IndexFile
from JunoPy.mag import MagData
from JunoPy.waves import WavesData
from JunoPy.library import find_ticks_between

In [ ]:
startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')

# startevent = dt.datetime.strptime('2017-083T13:01:29', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2017-083T13:02:02', '%Y-%jT%H:%M:%S')

# startevent = dt.datetime.strptime('2017-236T00:21:15', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2017-236T23:39:38', '%Y-%jT%H:%M:%S')

start = startevent - dt.timedelta(minutes=20)
end   = endevent + dt.timedelta(minutes=20)
    
if startevent.day != start.day:
    start = dt.datetime(startevent.year, startevent.month, startevent.day, 0, 0, 1)
    
if endevent.day != end.day:
    end = dt.datetime(endevent.year, endevent.month, endevent.day, 23, 59, 59)
    
print(start, end)
magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(startevent.strftime('%Y%j')))[0]
magdata = MagData(filename=magfile)
magdata.df = magdata.df.loc[start:end]

In [ ]:
def plot_data(magdata, start=None, end=None, outfolder=None, sevent=None, eevent=None):
    """Plot data between start and end."""
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(5, 1, 
                            gridspec_kw={'hspace': 0.1, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 11), 
                            dpi=150)
    
    magdata.calculate_bendback()
    
    axs[:] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag', 'alpha'], axs=axs[:5])
    
    for a in axs:
        a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
        a.axvline(sevent, ls='dotted', color='r') # [0.2, 0.2, 0.2])
        a.axvline(eevent, ls='dotted', color='r')
        
            
    annotations = ['a)', 'b)', 'c)', 'd)', 'e)']
    for i, a in enumerate(axs):
        a.text(0.96, 0.8, annotations[i], transform=a.transAxes, weight='bold')
        
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.92)
    
    if outfolder:
        fig.savefig(outfolder + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
        plt.close(fig)
    
    plt.show()
    fig.savefig('out_images/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
    
    # plt.close(fig)
    
plot_data(magdata, start=start, end=end, sevent=startevent, eevent=endevent)

In [ ]:
print(magdata.df.loc[startevent - dt.timedelta(minutes=20): startevent - dt.timedelta(minutes=10), 'alpha'].mean())